In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from matplotlib.ticker import FormatStrFormatter
import matplotlib.transforms

# Set seaborn theme
#sns.set_theme(style="white", palette="deep", font="Myriad Pro")
#sns.set_context("poster")


In [ ]:
# Configuration
REPLAY_FROM_CSV = True
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)


def load_preprocessed_long(csv_path=f"{DATA_DIR}/model_comparison_long.csv"):
    """Load the long-format dataframe."""
    df = pd.read_csv(csv_path)
    return df


# Load data
if REPLAY_FROM_CSV:
    df_long = load_preprocessed_long()


In [ ]:
# replace "Enformer Distilled" with "Distilled Enformer"
df_long["Model"] = df_long["Model"].replace("Enformer Distilled", "Distilled Enformer")


In [ ]:
task_groups = {
    "Histone": [
        "H2AFZ",
        "H3K9ac",
        "H3K27ac",
        "H3K4me1",
        "H3K4me2",
        "H3K4me3",
        "H3K27me3",
        "H3K36me3",
        "H3K9me3",
        "H4K20me1",
    ],
    "Enhancers": [
        "enhancers",
        "enhancers_types",
        "promoter_all",
        "promoter_tata",
        "promoter_no_tata",
    ],
    "Splice Sites": [
        "splice_sites_donors",
        "splice_sites_all",
        "splice_sites_acceptors",
    ],
}


In [ ]:
all_tasks = task_groups["Histone"] + task_groups["Enhancers"] + task_groups["Splice Sites"]
all_models = df_long["Model"].unique()

all_scores = {}
for model in all_models:
    model_scores = {}
    for task in all_tasks:
        model_scores[task] = df_long[
            (df_long["Model"] == model) & (df_long["Task"] == task)
        ]["Score"].values[0]
    all_scores[model] = model_scores


In [ ]:
# All tasks organized by row
#task_rows = [
#    histone_tasks[:5],  # First 5 H-tasks
#    histone_tasks[5:],  # Last 5 H-tasks
#    cre_tasks,  # Enhancers and promoters
#    splice_tasks,  # Splice sites
#]

# Updated color palette
custom_palette = {
    "BPNet": "#909090",
    "Nucleotide Transformer": "#A8C8E1",
    "Distilled Nucleotide Transformer": "#2E5C8A",
    "DNABERT-2": "#FFD4A3",
    "Distilled DNABERT-2": "#E67E22",
    "Enformer": "#82E0AA",
    "Distilled Enformer": "#27AE60",
    "Caduceus": "#D7BDE2",
    "Distilled Caduceus": "#8E44AD",
}


In [ ]:
model_groups = {
    "Nucleotide Transformer": ["Distilled Nucleotide Transformer", "Nucleotide Transformer", "BPNet"][::-1],
    "DNABERT-2": ["Distilled DNABERT-2", "DNABERT-2", "BPNet"][::-1],
    "Enformer": ["Distilled Enformer", "Enformer", "BPNet"][::-1],
    "Caduceus": ["Distilled Caduceus", "Caduceus", "BPNet"][::-1],
}


In [ ]:
def get_group_scores(group, model):
    scores = []
    for task in task_groups[group]:
        scores.append(all_scores[model][task])
    return np.array(scores)

def order_tasks(group, model):
    model = "Distilled " + model
    scores = get_group_scores(group, model)
    orders = np.argsort(scores)[::-1]
    return orders, np.array(task_groups[group])[orders]


In [ ]:
order_tasks("Histone", "Nucleotide Transformer")


In [ ]:
# group and sort elements
def prepare_group(group, model):
    orders, ordered_tasks = order_tasks(group, model)
    scores = {}
    for model_sub in model_groups[model]:
        scores[model_sub] = get_group_scores(group, model_sub)[orders]
    score_array = np.array(list(scores.values()))
    return ordered_tasks, score_array


In [ ]:
prepare_group("Histone", "Nucleotide Transformer")


In [ ]:
# plot group by group
def plot_model(group, model, ax, legend=True, connect=['all', 'main', 'none']):
    #xtick_offset = matplotlib.transforms.ScaledTranslation(0, 0.05, fig.dpi_scale_trans)
    ordered_tasks, scores = prepare_group(group, model)
    x_pos = np.arange(len(task_groups[group]))
    limits = [max(0, scores.min() - 0.1), scores.max() + 0.05]

    for i, task in enumerate(ordered_tasks):
        ax.plot([i,i], [limits[0],scores[:,i].min()], color="#B0B0B0", linestyle=":", linewidth=2, zorder=1)
        ax.plot([i,i], [scores[0,i], scores[1,i]], color="#B0B0B0", linestyle=":", linewidth=2, zorder=1)
        ax.plot([i,i], [scores[1,i], scores[2,i]], color="#B0B0B0", linestyle="-", linewidth=2, zorder=1)
    
    for j, model_sub in enumerate(model_groups[model]):
        ax.plot(x_pos, scores[j,:], 
            #markeredgecolor=custom_palette[model_sub], markeredgewidth=2, 
            #markerfacecolor="#FFFFFF",
            marker="o", markersize=10, color=custom_palette[model_sub],
            linestyle="",
            label=model_sub, zorder=3)
        if connect == 'all':
            ax.plot(x_pos, scores[j,:], color=custom_palette[model_sub], linestyle="-", linewidth=1, zorder=2)
        elif connect == 'main':
            if j == 2:
                ax.plot(x_pos, scores[j,:], color=custom_palette[model_sub], linestyle="-", linewidth=1, zorder=2)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(ordered_tasks, rotation=45, ha="right", va="top")
    ax.tick_params(axis='x', labelsize=10, pad=2, length=0)
    ax.set_xlim(-0.5, len(ordered_tasks) - 0.5)

    ax.set_yticks(np.arange(0.1, 1.0, 0.1))
    for y in ax.get_yticks():
        ax.plot([-1, len(ordered_tasks)], [y,y], color="#B0B0B0", linestyle="-", linewidth=0.3, zorder=0)
    ax.tick_params(axis='y', labelsize=10, pad=5, length=0)
    ax.set_ylim(*limits)

    ax.set_title(group, fontsize=12)
    if legend:
        ax.legend(loc="lower left", fontsize=10, markerscale=0.7)
    


In [ ]:
def lollipop_plot(all_scores, all_tasks, model, connect='none'):
    fig = plt.figure(figsize=(15,3.5))
    ax1 = plt.subplot(1,2,1)
    ax2 = plt.subplot(1,4,3)
    ax3 = plt.subplot(1,4,4)
    plt.subplots_adjust(top=0.85)
    plot_model("Histone", model, ax1, connect=connect)
    plot_model("Enhancers", model, ax2, legend=False, connect=connect)
    plot_model("Splice Sites", model, ax3, legend=False, connect=connect)
    fig.suptitle(model, fontsize=15)


In [ ]:
for model in model_groups.keys():
    lollipop_plot(all_scores, all_tasks, model, connect='none')


In [ ]:
df_long
